# 2. Byte Pair Encoding (BPE)

BPE is one of the most important **subword tokenization algorithms**. The key idea is beautifully simple:

> **Start with small units, repeatedly merge the most frequent adjacent pair, and build a vocabulary of useful subword units.**



The classic NLP formulation starts with characters and repeatedly merges the most frequent adjacent symbols. This creates a vocabulary containing characters **plus the newly learned merged units**. [Stanford University](https://www.web.stanford.edu/~jurafsky/slp3/ed3book.pdf?utm_source=chatgpt.com)

---

## 2.1 What is BPE?

**BPE = Byte Pair Encoding**

In NLP, BPE is used to learn a vocabulary of **subword units** from a training corpus.

Instead of deciding beforehand that:

```text
"playing" = one token
```

BPE learns useful pieces from the data:

```text
playing
   ↓
play + ing
```

Similarly:

```text
unhappiness
     ↓
un + happy + ness
```

The exact learned segmentation depends on the corpus and the merge rules.

### The basic process

```text
Raw text
   ↓
Initial small units
   ↓
Count adjacent pairs
   ↓
Find most frequent pair
   ↓
Merge the pair
   ↓
Add new token to vocabulary
   ↓
Repeat
```

This is a **bottom-up** approach to vocabulary construction. [LLM Book](https://llmbook.icsgen-ai.org/part-1-llm-building-blocks/module-01-foundations-nlp-text-representation/section-1.6.html?utm_source=chatgpt.com)

---

# 2.2 Start with Characters

Let's use a tiny corpus:

```text
low
lower
newest
widest
```

Initially, words are split into individual symbols:

```text
low    → l o w
lower  → l o w e r
newest → n e w e s t
widest → w i d e s t
```

Our initial vocabulary might therefore be:

```text
V = {l, o, w, e, r, n, s, t, i, d}
```

The important point is:

> **We begin with a small vocabulary of basic units instead of storing every complete word.**

The original BPE formulation then repeatedly examines adjacent symbol pairs across the corpus. [Stanford University](https://www.web.stanford.edu/~jurafsky/slp3/ed3book.pdf?utm_source=chatgpt.com)

---

# 2.3 Count Adjacent Pairs

Now count pairs such as:

```text
low
→ l o
→ o w

lower
→ l o
→ o w
→ w e
→ e r
```

Across the entire corpus, some pairs occur more frequently than others.

For example, suppose:

```text
(o, w) → 2 occurrences
(e, s) → 2 occurrences
(w, e) → 2 occurrences
```

The algorithm chooses the **most frequent pair**.

For the actual Stanford example, the first most frequent pair is `n e`, which occurs four times across the training corpus. [Stanford University](https://www.web.stanford.edu/~jurafsky/slp3/ed3book.pdf?utm_source=chatgpt.com)

---

# 2.4 Merge Operation

Suppose the most frequent pair is:

```text
n + e
```

BPE merges them:

```text
n + e → ne
```

So:

```text
newest
```

changes from:

```text
n e w e s t
```

to:

```text
ne w e s t
```

And `ne` is added to the vocabulary:

```text
V = {
    l, o, w, e, r, n, s, t, i, d,
    ne
}
```

This is called a **merge operation**.

Mathematically:

$$[
(a,b) \rightarrow ab
]
$$

where \(a\) and \(b\) are adjacent tokens.

---

# 2.5 Repeat the Process

Now we count pairs **again**.

This is important.

After creating:

```text
ne
```

the frequency statistics have changed.

Suppose the next most frequent pair is:

```text
ne + w
```

Then:

```text
ne + w → new
```

Now:

```text
newest
```

becomes:

```text
new e s t
```

And `new` is added to the vocabulary.

The Stanford example follows exactly this pattern: `ne` is first learned, followed by `new`. [Stanford University](https://www.web.stanford.edu/~jurafsky/slp3/ed3book.pdf?utm_source=chatgpt.com)

---

# 2.6 The Vocabulary Grows

This is the key idea behind **vocabulary creation**.

Initially:

```text
V₀ = {characters}
```

After one merge:

```text
V₁ = V₀ ∪ {ne}
```

After another:

```text
V₂ = V₁ ∪ {new}
```

After many merges:

```text
V_final =
{
  characters,
  common subwords,
  frequent word pieces,
  sometimes complete words
}
```

So BPE doesn't choose between:

> "character vocabulary" **OR** "word vocabulary"

Instead, it **builds a vocabulary between the two**.



---

# 2.7 Why Does This Produce Subwords?

Imagine the corpus repeatedly contains:

```text
playing
played
player
players
```

The algorithm may discover frequent pieces such as:

```text
play
```

and:

```text
ing
ed
er
s
```

So instead of storing:

```text
playing
played
player
players
```

as four independent vocabulary entries, we might obtain:

```text
play + ing
play + ed
play + er
play + er + s
```

This is where **subword units** come from.

### Important distinction

BPE does **not** explicitly understand morphology.

It doesn't know:

> "`play` is the root and `ing` is a suffix."

It simply learns frequent adjacent patterns.

Sometimes those patterns happen to correspond nicely to linguistic units.

That's an important interview point. ⭐

---

# 2.8 A Complete Toy Example

Let's make the process very small.

Corpus:

```text
low
lower
```

Initial representation:

```text
low
→ l o w

lower
→ l o w e r
```

Count pairs:

```text
l o → 2
o w → 2
w e → 1
e r → 1
```

Most frequent pair:

```text
l o
```

Merge:

```text
l + o → lo
```

Corpus becomes:

```text
low
→ lo w

lower
→ lo w e r
```

Vocabulary:

```text
{l, o, w, e, r, lo}
```

Count again:

```text
lo w → 2
w e  → 1
e r  → 1
```

Merge:

```text
lo + w → low
```

Corpus:

```text
low
→ low

lower
→ low e r
```

Vocabulary now contains:

```text
l
o
w
e
r
lo
low
```

Continue merging and we could eventually learn:

```text
lower
```

as a token too, if the vocabulary budget and corpus statistics make that merge worthwhile.

This illustrates an important property:

> **Frequent patterns tend to become larger tokens through repeated merges.**

---

# 2.9 Training vs Tokenization

This distinction is **very important**.

### During BPE training

We learn:

```text
merge 1: n + e → ne
merge 2: ne + w → new
merge 3: _ + r → _r
...
```

These merges are stored in an ordered merge table.

### During inference

For new text, we **do not relearn the vocabulary**.

Instead, we apply the learned merge rules.

For example:

```text
newword
```

is processed using the previously learned rules.

The BPE encoder applies the merges learned during training to test/new text. [Stanford University](https://web.stanford.edu/~jurafsky/slp3/ed3book.pdf?utm_source=chatgpt.com)

Think:

```text
TRAINING
Corpus
  ↓
Learn merge rules
  ↓
Vocabulary + merge table


INFERENCE
New text
  ↓
Apply learned merge rules
  ↓
Subword tokens
```

---

# 2.10 When Does BPE Stop?

We can't keep adding tokens forever.

Usually we specify a **target vocabulary size**.

For example:

```text
initial vocabulary = 1,000 tokens

target vocabulary = 30,000 tokens
```

Then BPE performs enough merges to reach the desired vocabulary size.

Conceptually:

$$[
|V_{final}| \approx |V_{initial}| + \text{number of merges}
]
$$

The exact implementation details vary between tokenizer implementations, especially between classic character-level BPE and modern byte-level BPE.

---

# 2.11 Why BPE Is Powerful

BPE gives us a useful balance:

| Approach | Vocabulary | Handles unseen words | Sequence length |
|---|---:|---:|---:|
| Word-level | Large | ❌ Poorly | Short |
| Character-level | Tiny | ✅ Very well | Very long |
| **BPE** | **Moderate** | **✅ Well** | **Moderate** |

For example:

```text
"unhappiness"
```

could potentially become:

```text
["un", "happi", "ness"]
```

rather than:

```text
["unhappiness"]
```

or:

```text
["u","n","h","a","p","p","i","n","e","s","s"]
```

So BPE gives us **compression + flexibility**.

---

# ⚠️ One Important Terminology Detail

You will encounter two related uses of the term **BPE**.

### Classic BPE

The original NLP formulation operates on characters/symbols and repeatedly merges frequent adjacent pairs. [Stanford University](https://www.web.stanford.edu/~jurafsky/slp3/ed3book.pdf?utm_source=chatgpt.com)

### Byte-level BPE

Modern tokenizers can start from **bytes** rather than Unicode characters.

This gives a base vocabulary capable of representing arbitrary text without requiring an `<UNK>` token for unseen Unicode characters. The trade-off is potentially longer sequences for some text. [d2l.smola.org](https://d2l.smola.org/slides/pytorch/chapter_recurrent-neural-networks/text-sequence.html?utm_source=chatgpt.com)

For now, keep the conceptual algorithm the same:

> **Start small → count adjacent pairs → merge the most frequent → repeat.**

We'll deal with **byte-level BPE** when implementation details become relevant.

---

# 🧠 BPE Mental Model

Remember this pipeline:

```text
              TRAINING
                 │
                 ▼
             Raw Corpus
                 │
                 ▼
        Split into basic units
                 │
                 ▼
       Count adjacent pairs
                 │
                 ▼
       Select most frequent pair
                 │
                 ▼
             Merge pair
                 │
                 ▼
       Add new token to vocabulary
                 │
                 ▼
             Repeat 🔁
                 │
                 ▼
       Learned vocabulary
       + merge rules
```

Then:

```text
              NEW TEXT
                 │
                 ▼
        Apply learned merges
                 │
                 ▼
        Subword token sequence
                 │
                 ▼
             Transformer
```

---

## Must Know ⭐

- **BPE = Byte Pair Encoding**
- Starts with small units.
- Counts **adjacent token pairs**.
- Merges the **most frequent pair**.
- Adds the merged unit to the vocabulary.
- Repeats this process.
- Produces a vocabulary containing characters/bytes and learned **subword units**.
- The learned merge rules are reused when tokenizing new text.
- BPE is **frequency-based**, not linguistically morphology-aware.

### Good to Know

- Vocabulary size controls how aggressively BPE merges tokens.
- Frequent patterns can eventually become whole-word tokens.
- Classic BPE and modern **byte-level BPE** are related but not identical implementations.

### Optional / Advanced

- Merge-rank tables
- Byte-level preprocessing
- Efficient pair-frequency updates
- BPE tokenizer implementation internals

---

## What you should remember

1. **BPE builds a subword vocabulary bottom-up.**
2. **The fundamental operation is:**  
   $$[
   a+b\rightarrow ab
   ]$$
3. At every iteration, BPE selects the **most frequent adjacent pair**.
4. Every merge creates a new vocabulary item.
5. Repeated merges transform characters/bytes into useful subword units.
6. **Training learns the merge rules; inference applies those rules.**
7. BPE doesn't understand morphology explicitly—it learns frequent patterns from data.

---